# CRANE Demo — Conformal Reliable Augmented Neural Framework

快速演示 CRANE 的核心功能：加载模型 → 多模态推理 → Conformal 不确定性量化

**要求**: 已安装依赖 (`pip install -r requirements.txt`)，已有训练好的模型 checkpoint

## 1. 环境准备

In [ ]:
import sys
import torch
import numpy as np

from utils.en_model import CRANEModel, CRANEModelMVE
from utils.conformal import *
from utils.data_loader import data_loader

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. 加载模型与数据

In [ ]:
from config import *
import os

# 加载数据（自动按 Plan C 分割）
try:
    train_loader, test_loader, es_loader, cal_loader = data_loader(
        BATCH_SIZE, 'mosi', seed=SEED
    )
    print(f'Train: {len(train_loader.dataset)} samples')
    print(f'Cal:   {len(cal_loader.dataset)} samples')
    print(f'Test:  {len(test_loader.dataset)} samples')
except FileNotFoundError:
    print('[WARNING] CMU-MOSI 数据未找到。请先运行: python convert_for_crane.py')
    print('跳过数据加载，展示 API 用法...')

## 3. Conformal Prediction API 演示

用模拟数据展示 5 种 conformal 方法的 API。真实推理流程与此完全相同。

In [ ]:
# 模拟数据：200 个校准样本 + 50 个测试样本
rng = np.random.RandomState(42)
n_cal, n_test = 200, 50
y_cal = rng.uniform(-3, 3, n_cal)
y_pred_cal = y_cal + 0.3 * rng.randn(n_cal)    # 预测有噪声
mc_std_cal = 0.1 + 0.3 * np.abs(y_cal)          # 不确定性随情感强度增大

y_test = rng.uniform(-3, 3, n_test)
y_pred_test = y_test + 0.3 * rng.randn(n_test)
mc_std_test = 0.1 + 0.3 * np.abs(y_test)

alpha = 0.10  # 90% 覆盖目标

In [ ]:
# (1) Split Conformal — 所有样本统一宽度
cp_split = SplitConformalPredictor()
cp_split.calibrate(y_cal, y_pred_cal)
lo_split, up_split = cp_split.predict(y_pred_test, alpha)
print(f'Split CP: coverage={compute_coverage(y_test, lo_split, up_split):.3f}, '
      f'width={np.mean(up_split - lo_split):.3f}')

In [ ]:
# (2) Adaptive Conformal — 宽度与 MC Dropout 不确定性成比例
cp_adapt = MCAdaptiveConformalPredictor()
cp_adapt.calibrate(y_cal, y_pred_cal, mc_std_cal)
lo_adapt, up_adapt = cp_adapt.predict(y_pred_test, mc_std_test, alpha)
print(f'Adaptive CP: coverage={compute_coverage(y_test, lo_adapt, up_adapt):.3f}, '
      f'width={np.mean(up_adapt - lo_adapt):.3f}')

# 验证自适应行为：不确定样本的区间更宽
widths_adapt = up_adapt - lo_adapt
corr_std_width = np.corrcoef(mc_std_test, widths_adapt)[0, 1]
print(f'Corr(σ_mc, width) = {corr_std_width:.3f} → 确认宽度随不确定性增大')

In [ ]:
# (3) MC Dropout RAW — 无校准的灾难性失败
lo_raw, up_raw = mc_dropout_interval(y_pred_test, mc_std_test, alpha)
print(f'MC RAW:   coverage={compute_coverage(y_test, lo_raw, up_raw):.3f} '
      f'(目标: ≥{1-alpha:.0%})')
print(f'→ 不校准的不确定性估计严重欠覆盖！')

In [ ]:
# (4) Mondrian Conformal — 按情感极性分组校准
groups_cal = sentiment_group(y_cal)
groups_test = sentiment_group(y_test)

cp_mond = MondrianConformalPredictor()
cp_mond.calibrate(y_cal, y_pred_cal, groups_cal)
lo_mond, up_mond = cp_mond.predict(y_pred_test, groups_test, alpha)
print(f'Mondrian CP: coverage={compute_coverage(y_test, lo_mond, up_mond):.3f}')

# 逐类覆盖
for g in ['negative', 'neutral', 'positive']:
    m = groups_test == g
    if m.sum() > 0:
        cov_g = compute_coverage(y_test[m], lo_mond[m], up_mond[m])
        print(f'  {g}: coverage={cov_g:.3f} (n={m.sum()})')

In [ ]:
# (5) Classification Conformal — 离散预测集
cp_cls = ClassificationConformalPredictor()
cp_cls.calibrate(y_cal, y_pred_cal)
pred_sets = cp_cls.predict(y_pred_test, alpha)

metrics = classification_set_metrics(y_test, pred_sets)
print(f'Classification CP: coverage={metrics["coverage"]:.3f}, '
      f'avg set size={metrics["avg_set_size"]:.1f}')

# 展示几个预测集
print('\n预测示例:')
for i in range(5):
    pred = y_pred_test[i]
    pset = pred_sets[i]
    true_c = map_to_7class(y_test[i])[0]
    covered = '✓' if true_c in pset else '✗'
    print(f'  ŷ={pred:+.2f} → Set={pset}  (true={true_c:+d}) {covered}')

## 4. 方法对比总结

In [ ]:
methods = {
    'Split CP': (lo_split, up_split),
    'Adaptive CP': (lo_adapt, up_adapt),
    'MC RAW': (lo_raw, up_raw),
    'Mondrian CP': (lo_mond, up_mond),
}

print(f'{"Method":<15} {"Coverage":>10} {"Avg Width":>10} {"Med Width":>10}')
print('-' * 48)
for name, (lo, up) in methods.items():
    cov = compute_coverage(y_test, lo, up)
    avg_w, med_w = compute_interval_width(lo, up)
    target = '✓' if cov >= 1 - alpha else '✗'
    print(f'{name:<15} {cov:>9.3f} {avg_w:>10.3f} {med_w:>10.3f}  {target}')

## 5. 真实验结果参考

以上演示使用模拟数据（i.i.d. Gaussian noise），仅展示 API 用法。

在真实的 CMU-MOSI 上训练后（`python run.py --seed 42`），预期结果如下：

| Method | Coverage | Med Width |
|:---|---:|---:|
| MC Dropout RAW | 34.4% ✗ | 0.70 |
| Split Conformal | 90.7% ✓ | 2.93 |
| Adaptive (MC Dropout) | 91.0% ✓ | 2.92 |
| Mondrian Conformal | 90.5% ✓ | 3.22 |
| MVE + Adaptive | 90.2% ✓ | 4.19 |
| Classification Set | 88.3% | 2.93 size |

详细结果见 `CRANE_SUMMARY.md`。